# Results Comparison: AbLang2-LoRA vs. Frozen AbLang2 + Ridge vs. Ridge/p-IgGen

Loads cross-validation results from both model notebooks and produces the final comparison tables and figures for the paper.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

LORA_DIR = Path("results/ablang2_developability")
RIDGE_DIR = Path("results/ablang2_ridge")
OUTPUT_DIR = Path("results/comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Approval Prediction Comparison

In [ ]:
# Load approval CV results from both models
lora_approval = pd.read_csv(LORA_DIR / "ablang2_fine_tune_LoRA_approval_cv_results.csv")
ridge_approval = pd.read_csv(RIDGE_DIR / "ablang2_ridge_approval_cv_results.csv")

def summarize_approval(df, name):
    return pd.DataFrame({
        "Model": [name],
        "AUPRC_mean": [df["AUPRC"].mean()],
        "AUPRC_std": [df["AUPRC"].std()],
        "AUROC_mean": [df["AUROC"].mean()],
        "AUROC_std": [df["AUROC"].std()],
    })

approval_table = pd.concat([
    summarize_approval(lora_approval, "AbLang2-LoRA"),
    summarize_approval(ridge_approval, "AbLang2 (frozen) + Ridge"),
]).reset_index(drop=True).round(3)

print("Table 1: Approval Prediction (5-fold CV)")
print("=" * 65)
print(approval_table.to_string(index=False))
approval_table.to_csv(OUTPUT_DIR / "approval_comparison.csv", index=False)

In [ ]:
# Paired bar chart: approval metrics
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)

for i, metric in enumerate(["AUPRC", "AUROC"]):
    ax = axes[i]
    models = approval_table["Model"]
    means = approval_table[f"{metric}_mean"]
    stds = approval_table[f"{metric}_std"]
    colors = ["#2d6a9f", "#e07b39"]
    bars = ax.bar(models, means, yerr=stds, capsize=5, color=colors, edgecolor="black", linewidth=0.5)
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.set_ylim(0, 1)
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + s + 0.02,
                f"{m:.3f}", ha="center", va="bottom", fontsize=9)
    ax.tick_params(axis="x", rotation=15)

fig.suptitle("Approval Prediction — 5-fold CV", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "approval_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Developability Regression Comparison

In [ ]:
lora_reg = pd.read_csv(LORA_DIR / "regression_cv_results.csv")
ridge_reg = pd.read_csv(RIDGE_DIR / "ablang2_ridge_regression_cv_results.csv")

# p-IgGen baselines from Graves et al.
RIDGE_PIGGEN_RHO = {
    "HAC": 0.621, "PR_Ova": 0.549, "PR_CHO": 0.496,
    "AC-SINS_pH7.4": 0.394, "HIC": 0.335, "Titer": 0.239,
    "SMAC": 0.207, "Purity": 0.180, "Tm2": -0.084, "SEC_%Monomer": -0.101,
}

def summarize_reg(df, name):
    return (
        df.groupby("target")["spearman_rho"]
        .agg(["mean", "std", "count"])
        .rename(columns={"mean": f"{name}_rho_mean", "std": f"{name}_rho_std", "count": f"{name}_n"})
        .round(3)
    )

lora_summary = summarize_reg(lora_reg, "LoRA")
ridge_summary = summarize_reg(ridge_reg, "Ridge")

regression_table = lora_summary.join(ridge_summary, how="outer")
regression_table["pIgGen_rho"] = regression_table.index.map(RIDGE_PIGGEN_RHO)
regression_table = regression_table.sort_values("LoRA_rho_mean", ascending=False).reset_index()

print("Table 2: Developability Regression — Spearman ρ (5-fold CV)")
print("=" * 80)
print(regression_table.to_string(index=False))
regression_table.to_csv(OUTPUT_DIR / "regression_comparison.csv", index=False)

In [ ]:
# Three-model horizontal bar chart
comp = regression_table.dropna(subset=["pIgGen_rho"]).sort_values("LoRA_rho_mean", ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, 6))
y = np.arange(len(comp))
w = 0.25

ax.barh(y + w, comp["LoRA_rho_mean"], w, label="AbLang2-LoRA (fine-tuned)", color="#2d6a9f")
ax.barh(y,     comp["Ridge_rho_mean"], w, label="AbLang2 (frozen) + Ridge", color="#e07b39")
ax.barh(y - w, comp["pIgGen_rho"],     w, label="Ridge / p-IgGen (baseline)", color="#b0c4de")

ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels(comp["target"].values, fontsize=10)
ax.set_xlabel("Spearman ρ (5-fold CV)", fontsize=11)
ax.set_title("Developability Metric Prediction\nThree-Model Comparison", fontsize=13)
ax.legend(fontsize=9, loc="lower right")
ax.set_xlim(-0.3, 0.8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "regression_3model_comparison.png", dpi=150)
plt.show()

## Summary Statistics

In [ ]:
# Mean improvement over p-IgGen baseline
comp_nums = regression_table.dropna(subset=["pIgGen_rho"]).copy()
comp_nums["lora_delta"] = comp_nums["LoRA_rho_mean"] - comp_nums["pIgGen_rho"]
comp_nums["ridge_delta"] = comp_nums["Ridge_rho_mean"] - comp_nums["pIgGen_rho"]

print("Per-target Δρ over p-IgGen baseline:")
print(comp_nums[["target", "LoRA_rho_mean", "Ridge_rho_mean", "pIgGen_rho", "lora_delta", "ridge_delta"]].to_string(index=False))
print(f"\nMean Δρ (LoRA):  {comp_nums['lora_delta'].mean():.3f}")
print(f"Mean Δρ (Ridge): {comp_nums['ridge_delta'].mean():.3f}")

comp_nums.to_csv(OUTPUT_DIR / "delta_over_baseline.csv", index=False)